# Breast Tumor Classification with PyTorch ResNet50

This notebook trains a PyTorch ResNet50 model to classify breast ultrasound (BUS) images as malignant or benign. The dataset provides `train.xlsx` and `test.xlsx`, so the model is trained only on the training split and evaluated only on the testing split.

## 1. Import Libraries and Set Paths

This cell imports the libraries used throughout the notebook and defines the dataset paths. The model uses the resized `224x224` image folder that was created earlier. PyTorch will use CUDA if a compatible GPU is available.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import random
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Store downloaded pretrained weights inside this project.
os.environ.setdefault('TORCH_HOME', '/data/ramialle/research_training/Research_Training/.torch')

# Dataset paths
DATASET_DIR = Path('/data/ramialle/datasets2/Dataset_BUSI_with_GT_Clean')
IMAGE_DIR = DATASET_DIR / 'images_224'
TRAIN_XLSX = DATASET_DIR / 'train.xlsx'
TEST_XLSX = DATASET_DIR / 'test.xlsx'

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
PATIENCE = 5
NUM_WORKERS = 2

# In the Excel files: 0 = malignant and 1 = benign.
CLASS_NAMES = {0: 'malignant', 1: 'benign'}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using images from: {IMAGE_DIR}')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'CUDA devices: {torch.cuda.device_count()}')

## 2. Read the Excel Train and Test Splits

This cell reads `train.xlsx` and `test.xlsx`. A small Excel reader is included so the notebook does not depend on `pandas` or `openpyxl` just to load the two split files.

In [ ]:
def _column_index(cell_reference):
    """Convert an Excel cell reference like A1 or B12 into a zero-based column index."""
    letters = ''.join(ch for ch in cell_reference if ch.isalpha())
    index = 0
    for char in letters:
        index = index * 26 + (ord(char.upper()) - ord('A') + 1)
    return index - 1


def read_xlsx_rows(path):
    """Read the first worksheet of a simple .xlsx file into a list of dictionaries."""
    namespace = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

    with ZipFile(path) as workbook_zip:
        shared_strings = []
        if 'xl/sharedStrings.xml' in workbook_zip.namelist():
            root = ET.fromstring(workbook_zip.read('xl/sharedStrings.xml'))
            for item in root.findall('main:si', namespace):
                text_parts = [node.text or '' for node in item.findall('.//main:t', namespace)]
                shared_strings.append(''.join(text_parts))

        sheet = ET.fromstring(workbook_zip.read('xl/worksheets/sheet1.xml'))
        raw_rows = []
        for row in sheet.findall('.//main:sheetData/main:row', namespace):
            values = []
            for cell in row.findall('main:c', namespace):
                column = _column_index(cell.attrib.get('r', 'A1'))
                while len(values) <= column:
                    values.append('')

                cell_type = cell.attrib.get('t')
                value_node = cell.find('main:v', namespace)
                inline_node = cell.find('main:is/main:t', namespace)

                if cell_type == 's' and value_node is not None:
                    value = shared_strings[int(value_node.text)]
                elif cell_type == 'inlineStr' and inline_node is not None:
                    value = inline_node.text or ''
                elif value_node is not None:
                    value = value_node.text or ''
                else:
                    value = ''

                values[column] = value
            raw_rows.append(values)

    header = [str(value).strip() for value in raw_rows[0]]
    rows = []
    for row in raw_rows[1:]:
        row = row + [''] * (len(header) - len(row))
        record = dict(zip(header, row))
        if record.get('Image'):
            record['Image'] = str(record['Image']).strip()
            record['Label'] = int(float(record['Label']))
            record['path'] = IMAGE_DIR / record['Image']
            record['class_name'] = CLASS_NAMES[record['Label']]
            rows.append(record)
    return rows


train_rows = read_xlsx_rows(TRAIN_XLSX)
test_rows = read_xlsx_rows(TEST_XLSX)

print(f'Training samples: {len(train_rows)}')
print(f'Testing samples: {len(test_rows)}')
print('Example training row:', train_rows[0])
print('Example testing row:', test_rows[0])

## 3. Verify Images and Class Counts

This cell checks that all images referenced by the Excel files exist and are exactly `224x224`. It also prints the benign/malignant counts in each split.

In [ ]:
def summarize_split(rows, split_name):
    labels = [row['Label'] for row in rows]
    counts = {CLASS_NAMES[label]: labels.count(label) for label in sorted(CLASS_NAMES)}
    print(f'{split_name} class counts:', counts)


def verify_images(rows, split_name):
    missing = [row['path'] for row in rows if not row['path'].exists()]
    if missing:
        raise FileNotFoundError(f'{split_name} has missing image files. First missing file: {missing[0]}')

    bad_sizes = []
    for row in rows:
        with Image.open(row['path']) as img:
            if img.size != IMG_SIZE:
                bad_sizes.append((row['Image'], img.size))

    if bad_sizes:
        raise ValueError(f'{split_name} has non-224x224 images. Examples: {bad_sizes[:5]}')

    print(f'{split_name}: all images exist and are {IMG_SIZE[0]}x{IMG_SIZE[1]}.')


summarize_split(train_rows, 'Train')
summarize_split(test_rows, 'Test')
verify_images(train_rows, 'Train')
verify_images(test_rows, 'Test')

## 4. Build PyTorch Datasets and DataLoaders

This cell converts the Excel rows into PyTorch datasets. Images are resized to `224x224` as a safety check and normalized using the official ImageNet preprocessing values for ResNet50.

In [ ]:
weights = ResNet50_Weights.DEFAULT
imagenet_normalize = weights.transforms()

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_normalize.mean, std=imagenet_normalize.std),
])

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_normalize.mean, std=imagenet_normalize.std),
])


class BreastTumorDataset(Dataset):
    """PyTorch dataset backed by the image paths and labels from the Excel split file."""
    def __init__(self, rows, transform=None):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        label = torch.tensor(row['Label'], dtype=torch.long)
        return image, label


train_dataset = BreastTumorDataset(train_rows, transform=train_transform)
test_dataset = BreastTumorDataset(test_rows, transform=test_transform)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=generator,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

images, labels = next(iter(train_loader))
print('Image batch shape:', tuple(images.shape))
print('Label batch shape:', tuple(labels.shape))

## 5. Create the PyTorch ResNet50 Model

This cell loads ResNet50 with pretrained ImageNet weights and replaces the original classification head with a two-class classifier for malignant vs benign BUS images.

In [ ]:
model = models.resnet50(weights=weights)

# Freeze the pretrained feature extractor for the first training stage.
for parameter in model.parameters():
    parameter.requires_grad = False

# Replace the ImageNet classification layer with a two-class BUS classifier.
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(num_features, 128),
    nn.ReLU(),
    nn.Dropout(p=0.25),
    nn.Linear(128, 2),
)
model = model.to(DEVICE)

train_labels = np.array([row['Label'] for row in train_rows])
class_counts = np.bincount(train_labels, minlength=2)
class_weights = len(train_labels) / (len(CLASS_NAMES) * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam((parameter for parameter in model.parameters() if parameter.requires_grad), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.2,
    patience=2,
    min_lr=1e-7,
)

total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

print('Model: ResNet50 with ImageNet pretrained weights')
print('Classifier head:', model.fc)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print('Class weights:', {CLASS_NAMES[i]: round(float(weight), 3) for i, weight in enumerate(class_weights)})

## 6. Train the Model

This cell trains the model on the training split and evaluates testing loss after every epoch using the testing split as validation data. Early stopping restores the model weights from the epoch with the lowest testing loss.

In [ ]:
def run_one_epoch(model, data_loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for images, labels in data_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += batch_size

    return total_loss / total, correct / total


history = {
    'train_loss': [],
    'test_loss': [],
    'train_accuracy': [],
    'test_accuracy': [],
}
best_test_loss = float('inf')
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
    test_loss, test_accuracy = run_one_epoch(model, test_loader, criterion, optimizer=None)
    scheduler.step(test_loss)

    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_accuracy'].append(train_accuracy)
    history['test_accuracy'].append(test_accuracy)

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train loss {train_loss:.4f}, train accuracy {train_accuracy:.4f} | '
        f'test loss {test_loss:.4f}, test accuracy {test_accuracy:.4f}'
    )

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_epoch = epoch
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}. Best epoch was {best_epoch}.')
            break

if best_state is not None:
    model.load_state_dict(best_state)
    model = model.to(DEVICE)

print(f'Best epoch by testing loss: {best_epoch} with testing loss {best_test_loss:.4f}')

## 7. Plot Training and Testing Loss

This cell plots the training loss and testing loss for each epoch. It also plots training and testing accuracy so underfitting or overfitting is easier to see.

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_ran, history['train_loss'], marker='o', label='Training loss')
plt.plot(epochs_ran, history['test_loss'], marker='o', label='Testing loss')
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'Best epoch {best_epoch}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Testing Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_ran, history['train_accuracy'], marker='o', label='Training accuracy')
plt.plot(epochs_ran, history['test_accuracy'], marker='o', label='Testing accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Testing Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluate with Accuracy, Recall, Precision, F1 Score, and Confusion Matrix

This cell evaluates the trained model on the testing set. Metrics are computed for both classes so that malignant and benign performance can be inspected separately.

In [ ]:
def predict_all(model, data_loader):
    model.eval()
    all_probabilities = []
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)

            all_probabilities.extend(probabilities[:, 1].cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_probabilities), np.array(all_predictions)


y_true, y_prob_benign, y_pred = predict_all(model, test_loader)

accuracy = np.mean(y_pred == y_true)
confusion = np.zeros((2, 2), dtype=int)
for true_label, pred_label in zip(y_true, y_pred):
    confusion[true_label, pred_label] += 1

print(f'Test accuracy: {accuracy:.4f}')
print('\nConfusion matrix rows=true, columns=predicted')
print('Labels:', [CLASS_NAMES[0], CLASS_NAMES[1]])
print(confusion)

print('\nPer-class metrics:')
macro_f1_values = []
for class_id, class_name in CLASS_NAMES.items():
    tp = confusion[class_id, class_id]
    fp = confusion[:, class_id].sum() - tp
    fn = confusion[class_id, :].sum() - tp

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    macro_f1_values.append(f1)

    print(f'{class_name:10s} precision={precision:.4f} recall={recall:.4f} f1={f1:.4f}')

print(f'\nMacro F1 score: {np.mean(macro_f1_values):.4f}')

## 9. Display Each Testing Image with True and Predicted Class

This final cell loops through the testing images and displays the image filename, true label, predicted class, and predicted benign probability. Misclassified images are highlighted in red titles.

In [ ]:
for row, probability, prediction in zip(test_rows, y_prob_benign, y_pred):
    true_name = CLASS_NAMES[row['Label']]
    predicted_name = CLASS_NAMES[int(prediction)]
    title_color = 'green' if prediction == row['Label'] else 'red'

    image = Image.open(row['path']).convert('RGB')
    plt.figure(figsize=(4, 4))
    plt.imshow(image, cmap='gray')
    plt.axis('off')
    plt.title(
        f"{row['Image']}\nTrue: {true_name} | Predicted: {predicted_name}\nBenign probability: {probability:.3f}",
        color=title_color,
        fontsize=10,
    )
    plt.show()